In [1]:
# ============================================================
# K-MEANS + NLP ANALYSIS
# Dataset: StatProfile_Carbon_Afforest_EN.csv
# ============================================================

# Cell 1: Import libraries

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLP resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("Libraries loaded successfully.")

Libraries loaded successfully.


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# ============================================================
# Cell 2: Load dataset
# ============================================================

file_path = "StatProfile_Carbon_Afforest_EN.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Shape: (48, 3)


,Year,Measure,Value
0,2000,Removals from the atmosphere due to afforestat...,0.96
1,2000,"Total emissions due to deforestation (CO2e/yr,...",18.00
2,2001,Removals from the atmosphere due to afforestat...,0.95
3,2001,"Total emissions due to deforestation (CO2e/yr,...",18.00
4,2002,Removals from the atmosphere due to afforestat...,0.96


In [3]:
# ============================================================
# Cell 3: Inspect dataset
# ============================================================

print("Columns:")
for i, col in enumerate(df.columns):
    print(i, ":", col)

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

Columns:
0 : Year
1 : Measure
2 : Value

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Year     48 non-null     int64  
 1   Measure  48 non-null     str    
 2   Value    48 non-null     float64
dtypes: float64(1), int64(1), str(1)
memory usage: 4.3 KB

Missing values:


Year       0
Measure    0
Value      0
dtype: int64

In [4]:
# ============================================================
# Cell 4: Identify text columns
# ============================================================

text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("Text columns found:")
print(text_columns)

# Display number of unique values in each text column
for col in text_columns:
    print(
        f"{col}: "
        f"{df[col].nunique()} unique values, "
        f"{df[col].isna().sum()} missing"
    )

Text columns found:
['Measure']
Measure: 2 unique values, 0 missing


In [5]:
# ============================================================
# Cell 5: Combine text columns
# ============================================================

# Combine all text columns into one NLP field.
# This makes the code work even if the dataset has multiple
# text/description columns.

if len(text_columns) == 0:
    raise ValueError("No text columns were found in the dataset.")

df["NLP_Text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

display(df[["NLP_Text"]].head())

,NLP_Text
0,Removals from the atmosphere due to afforestat...
1,"Total emissions due to deforestation (CO2e/yr,..."
2,Removals from the atmosphere due to afforestat...
3,"Total emissions due to deforestation (CO2e/yr,..."
4,Removals from the atmosphere due to afforestat...


In [6]:
# ============================================================
# Cell 6: NLP text cleaning
# ============================================================

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Keep alphabetic words
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    words = text.split()

    # Stopword removal and lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]

    return " ".join(words)

df["Clean_Text"] = df["NLP_Text"].apply(clean_text)

display(df[["NLP_Text", "Clean_Text"]].head())

,NLP_Text,Clean_Text
0,Removals from the atmosphere due to afforestat...,removal atmosphere due afforestation megatonnes
1,"Total emissions due to deforestation (CO2e/yr,...",total emission due deforestation megatonnes
2,Removals from the atmosphere due to afforestat...,removal atmosphere due afforestation megatonnes
3,"Total emissions due to deforestation (CO2e/yr,...",total emission due deforestation megatonnes
4,Removals from the atmosphere due to afforestat...,removal atmosphere due afforestation megatonnes


In [7]:
# ============================================================
# Cell 7: TF-IDF Vectorization
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(df["Clean_Text"])

print("TF-IDF matrix shape:", X.shape)

TF-IDF matrix shape: (48, 14)


In [8]:
 # ============================================================
# Cell 18: Save clustered dataset
# ============================================================

output_file = "StatProfile_Carbon_Afforest_EN_KMeans.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Clustered dataset saved as: {output_file}"
)

Clustered dataset saved as: StatProfile_Carbon_Afforest_EN_KMeans.csv
